# RegionPath f02 - 역량 갭 분석 기능 api, 데이터 테스트

In [5]:
from dotenv import load_dotenv
import os
import requests
import json
import pandas as pd
from IPython.display import display


## API 설정

In [6]:
# work24 OpenAPI 직무 정보 
load_dotenv()

API_KEY = os.environ.get('WORK24_API_KEY')
print(API_KEY)

BASE_URL = "https://www.work24.go.kr/cm/openApi/call/wk/callOpenApiSvcInfo215L01.do"


a65c5eb7-3439-41ad-a555-124aef9d95ed


In [24]:
def get_ncs_units_by_job(job_title: str, limit: int = 100) -> list[dict]:
    """
    직무명을 입력받아 관련 NCS 능력단위 목록을 반환합니다.

    Args:
        job_title: 직무명 (예: "백엔드 개발자", "데이터 분석가", "UI/UX 디자이너")
        limit: 반환할 최대 능력단위 수 (기본 5)

    Returns:
        각 항목 키:
            - unit_name      : 능력단위명
            - job_sdvn       : NCS 세분류 내 능력단위명
            - ablt_def       : 능력단위 정의
            - job_scfn       : NCS 세분류명
            - job_mcn        : NCS 중분류명
            - knwg_tchn_attd : 지식·기술·태도 레이블 리스트
    """
    params = {
        "authKey": API_KEY,
        "jobCont": job_title,
        "limit": limit,
        "returnType": "JSON",
    }
    response = requests.get(BASE_URL, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()

    if not isinstance(data, dict) or "result" not in data:
        raise ValueError(f"예상치 못한 응답 구조: {list(data.keys()) if isinstance(data, dict) else type(data)}")

    units = []
    for name, info in data["result"].items():
        unit = dict(info)
        unit["unit_name"] = name
        unit["knwg_tchn_attd"] = [
            k["knwg_tchn_attd_label"]
            for k in (info.get("knwg_tchn_attd") or [])
            if isinstance(k, dict)
        ]
        units.append(unit)
    return units

In [38]:
# 직무명 입력 → 관련 NCS 능력단위 조회
job_title = "백엔드"   # ← 여기에 직무명 입력

results = get_ncs_units_by_job(job_title, limit=5000)
print(f"[{job_title}] 관련 NCS 능력단위 {len(results)}개\n")

for i, unit in enumerate(results, 1):
    print(f"[{i}] {unit['unit_name']}")
    print(f"     세분류: {unit.get('job_scfn', '-')} / 중분류: {unit.get('job_mcn', '-')}")
    print(f"     정의  : {unit.get('ablt_def', '-')[:80]}...")
    knwg = unit.get("knwg_tchn_attd", [])
    print(f"     역량  : {', '.join(knwg[:3])}{'...' if len(knwg) > 3 else ''}")
    print()

[백엔드] 관련 NCS 능력단위 753개

[1] 디지털비지니스지원서비스 플랫폼 보안관리
     세분류: 통신서비스 / 중분류: 통신기술
     정의  : 디지털비즈니스지원서비스 플랫폼 보안관리란 디지털 비지니스 플랫폼을안전하게 서비스하기 위한 보안 요구사항을 파악하여 관리적, 물리적, 기술적 보안...
     역량  : 개인정보보호법, 개인정보보호법, 네트워크 QoS(Quality of Service)...

[2] 생육진단 시스템개발
     세분류: 스마트팜개발 / 중분류: 전자기기개발
     정의  : 생육진단 시스템개발이란 적용대상의 생육진단에 필요한 정보를 이용하여생육진단 알고리즘을 개발하는 능력이다....
     역량  : 시설 환경정보와 적용대상의 생육 변화에 대한 지식, 적용대상의 생육정보 종류와 수동, 자동수집 방법에 대한 지식, 적용대상의 생육정보 활용방법에 대한 지식...

[3] 자동차 조향장치 튜닝설계
     세분류: 자동차관리 / 중분류: 자동차
     정의  : 자동차 조향장치 튜닝설계란 사용자의 요청에 따라 스티어링 휠, 허브 장착 브라켓, 축, 기어, 휠, 타이어 등을 기획, 설계, 제작, 시험 할 ...
     역량  : 가공장비 및 가공법에 대한 지식, 기초적인 원 상태의 차량의 이해도, 도면 작성법...

[4] 송출시스템 운용
     세분류: 방송서비스 / 중분류: 방송기술
     정의  : 송출시스템 운용은 방송국내 콘텐츠를 시청자(청취자)에게 전달하기 위해서 콘텐츠 인제스트, 자동송출시스템관리, 네트워크 관리, 비상 조치할 수있는...
     역량  : API에 대한 지식, Demand Service 지식(VOD, GOD 등), Docsis Specification 지식...

[5] 정보기술 프로세스관리 전략 수립
     세분류: 정보기술전략·계획 / 중분류: 정보기술
     정의  : 정보기술 프로세스관리 전략 수립이란 조직의 경영비전·경영전략을 참조해서 고객과 시장을 공략하기 위해 사업, 

In [28]:
COLUMN_MAP = {
    "unit_name"     : "능력단위명",
    "ablt_def"      : "능력단위 정의",
    "job_scfn"      : "NCS 세분류명",
    "job_mcn"       : "NCS 중분류명",
    "knwg_tchn_attd": "지식·기술·태도",
}

df = (
    pd.DataFrame(results)
    .rename(columns=COLUMN_MAP)
    .assign(**{"지식·기술·태도": lambda d: d["지식·기술·태도"].apply(lambda v: "\n".join(v) if isinstance(v, list) else v)})
)
cols = [c for c in COLUMN_MAP.values() if c in df.columns]
display(df[cols])


,능력단위명,능력단위 정의,NCS 세분류명,NCS 중분류명,지식·기술·태도
0,스마트문화앱콘텐츠 유통채널 관리,스마트문화앱콘텐츠 유통 채널 관리란 스마트문화앱콘텐츠의 유통 활성화를 위하여 앱 유...,문화콘텐츠유통·서비스,문화콘텐츠,스마트문화앱 유통채널 분석자료 검토 및 평가 방법\n스마트문화앱 유통채널 분석자료 ...
1,비주얼 아이데이션 전개,비주얼 아이데이션 전개란 프로젝트의 디자인 콘셉트에 대한 효과적인 생각들을 시각적으...,디자인,디자인,논리적인 디자인 전개기법\n디자인 요소 표현기법\n색채 이론\n각종 스케치 도구 사...
2,생육진단 시스템개발,생육진단 시스템개발이란 적용대상의 생육진단에 필요한 정보를 이용하여생육진단 알고리즘...,스마트팜개발,전자기기개발,시설 환경정보와 적용대상의 생육 변화에 대한 지식\n적용대상의 생육정보 종류와 수동...
3,매니지먼트,매니지먼트란 아티스트의 연예활동을 관리하고 보좌하며 스타마케팅을 통하여 수익을 창출...,문화콘텐츠제작,문화콘텐츠,각종 인력관련 계약 지식\n계약에 대한 기본 지식\n공연과 관련된 각종 법률 지식\...
4,반도체 장비 시스템 소프트웨어 개발,반도체 장비 시스템 소프트웨어 개발이란 운용하고자 하는 주요부 및 요소기술 장치에 ...,반도체개발,전자기기개발,
...,...,...,...,...,...
270,의료정보 전사,의료정보 전사란 의사의 구술 또는 녹음된 진료내용이나 각종 검사결과가신속하고 정확하...,의료기술지원,보건,"관련 법령 (의료법, 전자의무기록시스템 인증제도 운영에 관한 고시)\n의료법과 관련..."
271,로봇 시스템 통합 소프트웨어 개발,로봇 시스템 통합 소프트웨어 개발이란 여러 대의 로봇 및 서버가 연동하여 동작할 때...,로봇개발,전자기기개발,
272,인터넷멀티미디어방송 응용서비스개발,응용서비스개발이란 인터넷멀티미디어방송에서 요구되는 다양한 부가서비스 일체를 개발하는...,방송플랫폼기술,방송기술,개발 설계서에 대한 지식\n기술규격\n모델별 소프트웨어 개발론\n개발계획서 작성 능...
273,SW아키텍처 이행,SW아키텍처 이행이란 아키텍처 설계에서 정의된 구조와 요소 기술에 따라 최종 산출물...,정보기술개발,정보기술,
